In [2]:
import pandas as pd
from string import Template
from pathlib import Path

DATA_FILE = Path("../matharc/data/qwen-0s-prompts.csv")
TEMPLATE_FILE = Path("gemma_zeroshot_template.txt")
OUTPUT_FILE = Path("data/gemma-multilingual-zero-prompts.csv")


In [4]:
df = pd.read_csv(DATA_FILE)
template_text = TEMPLATE_FILE.read_text(encoding="utf-8")
tmpl = Template(template_text)

In [5]:
language_map = {
    "Sri Lankan": ["Sinhala", "Tamil"],
    "Indian": ["Hindi", "Punjabi", "Marathi", "Odia"],
    "Albanian": ["Albanian"],
}

In [6]:
filled_rows = []

for idx, row in df.iterrows():
    addressing = row["addressing"]

    if addressing not in language_map:
        raise ValueError(f"Unknown addressing value: {addressing}")

    for language in language_map[addressing]:
        row_dict = row.to_dict()
        row_dict["language"] = language  # add new column for template use

        mapping = {}
        for k, v in row_dict.items():
            if pd.isna(v):
                mapping[k] = ""
            else:
                mapping[k] = str(v)

        try:
            out = tmpl.substitute(mapping)
        except KeyError as e:
            raise KeyError(f"Missing placeholder value for: {e.args[0]}") from None

        row_dict["multilingual_prompt"] = out
        filled_rows.append(row_dict)

df_out = pd.DataFrame(filled_rows)


In [7]:
print(df_out['multilingual_prompt'].iloc[149])

Your task is to create exactly one math word problem in Punjabi for a specific topic tailored to the NCERT curriculum, following a strict set of rules.

CONSTRAINTS:
**Cumulative Learning:** The curriculum's topics are ordered from simple to complex. A question for any given topic **may only** use concepts from that topic and all preceding topics. Must **NOT** use any concepts from topics that appear later in the list. This ensures questions build on prior knowledge.
**Grade Level:** The problem must be suitable for a Grade 3 student.
**Cultural Context:** The problem must feel authentic to a Indian child.

ORDERED GRADE 3 TOPIC LIST:
1. Counting : reads and writes numbers up to 999 using place value
2. Compare numbers : compares numbers up to 999 for their value based on their place value
3. addition : solves simple daily life problems using addition and subtraction of three digit numbers with and without regrouping, sums not exceeding 999
4. subtraction : solves simple daily life pro

In [8]:
df_out.to_csv(OUTPUT_FILE, index=False)

print("Done. Wrote:", OUTPUT_FILE)

Done. Wrote: data/gemma-multilingual-zero-prompts.csv
